# Phase 3 — Data Preparation

**Inputs:** `data/interim/accepted_curated.parquet` (from Phase 2 — 1,373,915 rows × 29 cols).

**Outputs:** `data/processed/train.parquet`, `data/processed/test.parquet` — model-ready, time-split, dtypes locked.

**Scope of “prep” here.** Two things go in this phase, and two things don’t.

*In scope* — changes that are model-agnostic and lossless:
1. Type coercion (strings that are really numbers; dates that are really dates).
2. Feature engineering that’s purely deterministic from existing columns (`fico_mean`, `credit_history_years`).
3. Semantic imputation (`mort_acc` NaN → 0 because zero is what the column literally measures when missing).
4. Time-based train/test split. This is a data-prep decision, not a modeling one — you don’t want a random shuffle leaking 2018 loans into the training data.

*Out of scope* — deferred to Phase 4:
1. **Statistical imputation** (`emp_length`, `dti`, `revol_util` medians). These have to live *inside* a sklearn `Pipeline` so the median is computed per CV fold. Doing it here would leak the test-set median into training.
2. **Encoding & scaling.** XGBoost wants integer-encoded categoricals; logistic regression wants one-hot + standardized numerics. Same data, two encodings — don’t lock one in here.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
sys.path.insert(0, str(PROJECT / 'src'))

from load import load_curated
from prepare import (
    clean, time_split, prepare_and_save, load_processed,
    TRAIN_MAX_YEAR, TEST_MIN_YEAR, CATEGORICAL_COLS,
    TRAIN_PARQUET, TEST_PARQUET,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

In [ ]:
df = load_curated()
print(f'Loaded: {df.shape}')
print(f'Mem:    {df.memory_usage(deep=True).sum()/1e6:.0f} MB')
df.dtypes.to_frame('dtype').T

## 1. Type coercion — strings that are really numbers

Three columns came out of the CSV as text but encode quantitative information:

| Column | Raw values | Target |
|---|---|---|
| `term` | ` ' 36 months'`, `' 60 months'` (note leading space) | `int8` ∈ {36, 60} |
| `emp_length` | `'< 1 year'`, `'1 year'`, ..., `'10+ years'`, `NaN` | `Float32` ∈ {0, 1, ..., 10}, NaN preserved |
| `earliest_cr_line` | `'Aug-2003'` | year as `Int16` |

`emp_length` is the only one where NaN is information — 6.5% of borrowers don’t report it, and “unwilling/unable to report” itself correlates with risk. We map the values *and* create `emp_length_missing` as a binary indicator. The pipeline can later median-impute the value column while still seeing the missingness signal.

In [ ]:
for c in ['term', 'emp_length', 'earliest_cr_line']:
    print(f'{c}: {df[c].value_counts(dropna=False).head(12).to_dict()}')
    print()

## 2. Engineered features

Two new columns, both lossless functions of existing ones:

- **`fico_mean = (fico_range_low + fico_range_high) / 2`** — LendingClub publishes FICO as a range; the gap is always 4 points, so `low` and `high` are perfectly collinear. Carrying both adds multicollinearity for linear models with zero information gain. Replace both with `fico_mean`.
- **`credit_history_years = issue_year - earliest_cr_line_year`** — raw `earliest_cr_line` is a date that’s only meaningful *relative to* the issue date. The difference is the credit-bureau “age” at application time, which is what underwriters actually look at.

Both are computable from the application data only — no leakage.

## 3. Semantic imputation (do here, not in the pipeline)

Two count columns get filled with 0 in this phase, *not* with a median in the pipeline:

- **`mort_acc`** (2.2% missing): “number of mortgage accounts.” When the bureau doesn’t report a value, the modal interpretation is “no mortgage accounts on file,” not “unknown count.” Median would push these toward the population mean (~1 mortgage) and overstate borrower stability.
- **`pub_rec_bankruptcies`** (0.06% missing): same logic. Zero bankruptcies is the sensible default.

The other small-NaN columns (`dti`, `revol_util`, `emp_length`) carry no obvious zero-semantics, so we leave them NaN for the pipeline’s `SimpleImputer(strategy='median')` to handle inside CV.

## 4. Apply everything via `prepare.clean()`

All of the above is implemented in `src/prepare.py`. The notebook just calls it — the source-of-truth is the module so Phase 4 sees the same data shape.

In [ ]:
cleaned = clean(df)
print(f'Before: {df.shape}  mem={df.memory_usage(deep=True).sum()/1e6:.0f} MB')
print(f'After:  {cleaned.shape}  mem={cleaned.memory_usage(deep=True).sum()/1e6:.0f} MB')
cleaned.dtypes.to_frame('dtype')

In [ ]:
# Remaining missingness — should be only the deferred columns.
miss = (cleaned.isna().mean() * 100).sort_values(ascending=False)
print('Columns still missing (intentionally — statistical imputation happens in Phase 4 pipeline):')
print(miss[miss > 0].round(3).to_string())

In [ ]:
# Sanity checks on engineered features.
print('credit_history_years:')
print(cleaned['credit_history_years'].describe().round(2).to_string())
print(f'\nfico_mean range: [{cleaned["fico_mean"].min()}, {cleaned["fico_mean"].max()}]')
print(f'emp_length_missing share: {cleaned["emp_length_missing"].mean()*100:.2f}%')

## 5. Time-based train/test split

**Why not a random split?** Loans are issued in time order and macro conditions move. A random shuffle would let the model peek at 2018 conditions while predicting 2010 loans — cheating in the wrong direction. In production this model scores *next year’s* applications, so the holdout should imitate that: train on the past, test on the future.

**Why the 2016 / 2017 cut?** From Phase 2’s year-by-year counts:

- Cumulative through 2016 = 1,129,956 rows (82.2%) — train
- 2017–2018 = 243,959 rows (17.8%) — test

This is the conventional 80/20 split *and* it gives the test set a full two-year horizon, which catches the late-cycle default uptick Phase 2 flagged.

**Caveat on the test default rate.** Because 2018 loans have had less time to mature, their charge-off rate is biased *down* (survivorship). The test set’s observed default rate (~27%) is therefore a lower bound — the real-world performance gap will be at least this large, probably larger.

In [ ]:
train, test = time_split(cleaned)

print(f'TRAIN  issue_year ≤ {TRAIN_MAX_YEAR}:  {len(train):>10,} rows '
      f'({len(train)/len(cleaned)*100:.1f}%)  default rate: {train["default"].mean()*100:.2f}%')
print(f'TEST   issue_year ≥ {TEST_MIN_YEAR}:  {len(test):>10,} rows '
      f'({len(test)/len(cleaned)*100:.1f}%)  default rate: {test["default"].mean()*100:.2f}%')

# Hard assertion: no temporal overlap between splits.
assert train['issue_year'].max() < test['issue_year'].min(), 'time-split leakage'
print('\nNo temporal overlap between splits: OK')

In [ ]:
# Distribution of years within each split — confirms the split is what we think it is.
print('Train year distribution:')
print(train['issue_year'].value_counts().sort_index().to_string())
print('\nTest year distribution:')
print(test['issue_year'].value_counts().sort_index().to_string())

In [ ]:
# Categorical level coverage — do any test categories appear that weren't in train?
# (XGBoost / OHE both handle this, but we want to know.)
for c in CATEGORICAL_COLS:
    new_in_test = set(test[c].dropna().unique()) - set(train[c].dropna().unique())
    if new_in_test:
        print(f'{c}: {len(new_in_test)} new levels in test: {sorted(new_in_test)[:5]}')
    else:
        print(f'{c}: all test levels present in train')

## 6. Save processed parquets

Phase 4 reads these via `prepare.load_processed()` — no need to re-run cleaning.

In [ ]:
TRAIN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
train.to_parquet(TRAIN_PARQUET, index=False)
test.to_parquet(TEST_PARQUET, index=False)
print(f'Wrote {TRAIN_PARQUET}  ({TRAIN_PARQUET.stat().st_size/1e6:.1f} MB)')
print(f'Wrote {TEST_PARQUET}  ({TEST_PARQUET.stat().st_size/1e6:.1f} MB)')

# Round-trip check — dtypes survive the parquet write.
tr2, te2 = load_processed()
assert tr2.shape == train.shape and te2.shape == test.shape
print('\nRound-trip OK.')

## 7. Phase 3 findings

1. **Working dataset:** 1,373,915 rows × 27 cols, **162 MB** in memory (down from 695 MB — 4× reduction from categorical/integer dtypes).
2. **Engineered features:** `fico_mean`, `credit_history_years` (median 15 yrs), `emp_length_missing` (5.88%).
3. **Remaining missingness, all deferred to Phase 4 pipeline:** `emp_length` (5.88%), `revol_util` (0.07%), `dti` (0.03%), small tails on `delinq_2yrs`/`credit_history_years`/`pub_rec`/`total_acc`/`open_acc` (~0.002% each).
4. **Time split:** train 1,129,956 rows @ **20.25%** default; test 243,959 rows @ **27.20%** default. The 7-point gap is real concept drift — macroeconomic conditions plus survivorship bias on the recent cohort. Phase 4 must report performance on **test**, not on a cross-validated estimate that smooths over this shift.
5. **Categorical level coverage:** no surprise levels in test that train didn’t see (verified above). One-hot or ordinal encoding is safe.

---

### Check-for-understanding (answer before Phase 4)

1. Why is `mort_acc.fillna(0)` done here in Phase 3, but `dti` is left NaN for the pipeline?
2. Why did we collapse `fico_range_low` + `fico_range_high` into a single `fico_mean` instead of keeping both?
3. The train default rate is 20.25% but the test default rate is 27.20%. What’s the most likely reason, and which model-evaluation metric would *understate* this drift most badly?
4. If you had used `train_test_split(..., random_state=42)` instead of a time-based split, what specific failure mode would show up in production scoring?
5. We kept `emp_length` NaN in this phase even though we know the median. What’s wrong with `cleaned['emp_length'].fillna(cleaned['emp_length'].median())` *right here* in the notebook?